In [1]:
import numpy as np
import pandas as pd
import torch
from torchvision import datasets , transforms 
from torch.utils.data import DataLoader

import warnings
warnings.filterwarnings("ignore")

In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [3]:
dataset = datasets.ImageFolder(root='xray', transform=transform, target_transform=None)

In [4]:
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [5]:
print("Mubarak ho! Total images mili:", len(dataset))
print("Classes ke naam:", dataset.classes)

Mubarak ho! Total images mili: 624
Classes ke naam: ['NORMAL', 'PNEUMONIA']


In [6]:
import torch.nn as nn
from torchvision import models

# 1. Pre-trained ResNet18 model ko download karna 
model = models.resnet18(pretrained=True)

# 2. purane layers ke weights ko freeze karna (taaki humein sirf last layer train karni pade)
for param in model.parameters():
    param.requires_grad = False

# 3. last layer ko change karna taaki woh hamari 2 classes (normal aur PNEUMONIA) ke liya result de skae 
num_features = model.fc.in_features
model.fc=nn.Linear(num_features,2)

print("Model successfully taiyar ho gaya hai ")
print(model)

Model successfully taiyar ho gaya hai 
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=Tr

In [7]:
import torch.optim as optim 

# 1. loss Function (Binary / Multi_class classification ke liya crossEntropyLoss best hai)
Criterion = nn.CrossEntropyLoss()

# 2. Optimizer (Adam optimizer model ke weights ko update karta hai)
# Yahan hum sirf model.fc.parameters() ko train kar rahe hain kyunki baaki sab freeze hain

optimizer = optim.Adam(model.fc.parameters(),lr=0.001)

print("loss function aur optimizer ready ho chuke hain")

loss function aur optimizer ready ho chuke hain


In [8]:
import sys
print(sys.executable)

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\python.exe


In [9]:
!pip install "numpy<2"


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
model.train()

# Epoch ka matlab hota hai ki model kitni baar saari X-ray images ko ek-ek karke dekhega
num_epochs=3
print("Training getting starting, it's take some time ")

for epoch in range(num_epochs):
    running_loss = 0.0
    # DataLoader se 32-32 images ke batches uthana
    for i, (images, lables) in enumerate(data_loader):

        # 1. Purane gradients ko zero karna
        optimizer.zero_grad()

        # 2. Images ko model ke andar pass karna (Forward pass)
        outputs = model(images)

        # 3. Check karna ki model ne kitni galti ki (Loss calculate karna)
        loss = Criterion(outputs,lables)

        # 4. Galti ko pehchankar model ke weights ko update karna (Backward pass)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()


    print(f"Epoch[{epoch+1}/{num_epochs}] poora ho gaya! loss: {running_loss/len(data_loader):.4f}")
print("\nCongratulations you x_ray model successfully train ho gaya hai ")

Training getting starting, it's take some time 
Epoch[1/3] poora ho gaya! loss: 0.5845
Epoch[2/3] poora ho gaya! loss: 0.3163
Epoch[3/3] poora ho gaya! loss: 0.2476

Congratulations you x_ray model successfully train ho gaya hai 


In [11]:
# model ke weights ko file main save karna 
torch.save(model.state_dict(),"xray_resnet18_model.pth")


In [12]:
from PIL import Image
import torch
from torchvision import transforms

#1 model ko evaluation(testing) mode par set karna
model.eval()

# 2 Apne xray folder se kisi bhi ek image ka pata yahan dein 
# mmisal ke taur par Normal ya pneumonia folder ke kisi image ka path check karke daal sakte hain 
images_path = "Xray/NORMAL/IM-0001-0001.jpeg" # yahan apni image ka check karke daal sakte hain

# 3 Image ko load karke resize aur tensor mein badalna 
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

image = Image.open(images_path).convert('RGB')
image_tensor = transform(image).unsqueeze(0)

with torch.no_grad():
    outputs= model(image_tensor)
    _, predicted= torch.max(outputs,1)

class_names = dataset.classes
result = class_names[predicted.item()]

print(f"Is X-ray ka result hai: {result}")

Is X-ray ka result hai: NORMAL
